# LAMb on Colab

Latent Arithmetic Machine. This notebook sets up the repo, **verifies the GPU path**,
and runs the open experiments.

The GPU path matters: everything in this project was developed on 4 CPU cores, so
cell 3 exercises the two failure modes that only appear on CUDA (a device mismatch in
the residue algebra's lookup tables, and fp16 underflow in distributions over small
rings, where a lost tail decodes to a *different integer* rather than a near miss).
Run it before trusting any result.

Runtime → Change runtime type → **A100** (or L4). High-RAM is not needed.


## 1. What hardware did we get?

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

### Sizing for your card

The models here are tiny, so VRAM is rarely the limit — but **kernel launch latency
is**. At 283k parameters a GPU barely beats a CPU because every kernel is too small
to fill it. Raise `d_model` and `batch` together or the card idles.

| card | suggested | note |
| --- | --- | --- |
| RTX 3050 4GB | `--d-model 768 --batch-size 256` | ~1.4 GB; Ampere, so use bf16 |
| RTX 3050 4GB (max) | `--d-model 1024 --batch-size 512` | ~2.5 GB, still fits |
| A100 40GB | `--d-model 1024 --batch-size 1024` | headroom for a 100M core |

Two things that bite on a small card specifically:

* `lamb.study` spawns one **process per worker**, and each carries its own CUDA
  context (~300–500 MB before any parameters). Four workers OOMs a 4 GB card. The
  study now clamps to `workers=1` automatically when CUDA is present, but pass
  `--workers 1` explicitly if you script around it.
* For the bridge, **cache the encoder's outputs in a separate pass**, then free the
  GPU before training. The encoder never has to coexist with the optimiser, so its
  size stops mattering — which is what makes a frozen encoder practical on 4 GB at
  all.

## 2. Install\n\nThe Rust kernels are optional — there is a pure-Python fallback, and `USING_RUST=False` changes speed, not results.\n\n**On Windows with a CUDA card**, PyPI's `torch` wheel is CPU-only; add `--index-url https://download.pytorch.org/whl/cu124` to get the GPU build. On Linux the default wheel already includes CUDA.

In [ ]:
BRANCH = "claude/lamb-autoregressive-transformer-lyl2hv"
!git clone -q --branch $BRANCH https://github.com/IlumCI/FinLLM.git /content/lamb || true
%cd /content/lamb
# uv resolves the pinned graph in well under a second and parallelises the
# downloads; torch dominates either way, so expect ~2x on a cold install and
# near-instant on a warm cache. The reason to prefer it here is uv.lock, which
# pins torch to an exact version across this machine, Colab and your own box --
# torch changes kernel selection and reduction order between minor versions, and
# on a model this small that moves accuracy, which would read as a finding.
!pip -q install uv
!uv pip install -q --system -e . pytest
# optional: the Rust kernels. Measured at ~1.5% of a training step, so this is
# about the self-play verifier's throughput, not training speed.
!uv pip install -q --system maturin && (cd rust && maturin develop --release -q) || echo "using the Python fallback"
import lamb._native as N; print("rust kernels:", N.USING_RUST)

## 3. Verify the GPU path

This is the cell that matters. It checks the algebra is still **exact** on CUDA and
under half precision — the two bugs that are unreachable on CPU.

In [ ]:
import torch
from lamb.algebra import ResidueSystem, ResidueAlgebra
from lamb.alu import LatentALU

dev = "cuda" if torch.cuda.is_available() else "cpu"
s, alg = ResidueSystem(), ResidueAlgebra(ResidueSystem())

# (a) lookup tables must follow the DATA to the GPU, not the constructor
m = LatentALU(d_model=64).to(dev)
a = s.onehot([1234], device=dev); b = s.onehot([5678], device=dev)
got = s.decode(m.alg.compose(a, b, "*", logits=False))
print(f"exact multiply on {dev}: {got[0]} == {1234*5678} ->", got[0] == 1234*5678)

# (b) fp16 inputs (what autocast hands it) must not corrupt the ring
got = s.decode(alg.compose(a.half(), b.half(), "*", logits=False))
print("exact under fp16 inputs:", got[0] == 1234*5678)

# (c) composition stays exact at magnitudes nothing was trained on
import random; rng = random.Random(0); bad = 0
for _ in range(2000):
    x, y = rng.randint(-99999, 99999), rng.randint(-999, 999)
    for op, want in (("+", x+y), ("-", x-y), ("*", x*y)):
        if s.representable(want) and alg.compose_exact(x, y, op) != want: bad += 1
print("exactness over 2000 random triples: ", bad, "errors")

## 4. Test suite

In [ ]:
!python -m pytest tests/ -q -p no:cacheprovider

## 5. The open experiments

Every number in `docs/ROADMAP.md` came from 4 CPU cores at 283k parameters. On an
A100 the whole backlog below is roughly **1–2 hours**. Run them in this order — the
first two are open scientific questions, not confirmations.

**`LAMB_DEVICE=cuda` overrides every config's device without editing code.**

### 5a. Did the ALU fail, or was it just too small?

Multi-digit residues never started learning at 283k params — flat at chance for 1200
steps even with the whole budget on one width. That is equally consistent with "the
design is wrong" and "the model is too small to induce a periodic digit map". This
settles it. A negative here at 20x the width is a real negative.

In [ ]:
%env LAMB_DEVICE=cuda
!python -m lamb.lotus --steps 6000 --batch-size 256 --d-model 512 \
    --depth 2 --digits 3 --n-latent 16 --trace-coef 0 --amp

### 5b. Does outcome-only program induction scale?

**Already answered at depth 2 on CPU, and the answer was yes**: with `program_coef=0`
— no gold program anywhere — the model reached **1.000 held-out answer accuracy**
while matching the generator's program on *zero* instructions. That is not a
contradiction: 48 distinct three-instruction programs compute `(a+b)+(c+d)` exactly,
and the grammar emits one of them, so `canonical_acc` measures conformity to its form
while `answer_acc` measures correctness.

What is open is whether it **scales**. Depth 2 is three instructions over four
operands — a small program space. Depth 3 is seven instructions over eight, and
depth 4 is fifteen over sixteen. Raise `depth` below; that is the experiment.

Gradients reach the pointer heads *through exact arithmetic*, which is what an
external Python interpreter cannot offer — PAL/Program-of-Thought can only imitate a
gold program or reinforce, and RL on this latent block was measured inert.


In [ ]:
import torch, time
from lamb import ArithmeticTokenizer, LotusConfig
from lamb.config import ModelConfig
from lamb.regmachine import RegMachineTrainer

for arm, pc in (("supervised", 1.0), ("answer-only", 0.0)):
    cfg = LotusConfig(steps=4000, batch_size=256, n_latent=16, loops=3, depth=2,
                      trace_coef=0.0, alu_coef=0.0, use_boundaries=False,
                      switch_coef=0.0, device="cuda", amp=True,
                      alu_moduli=(16,25,27,11,37))
    mcfg = ModelConfig(d_model=512, n_heads=8, d_ff=1024, n_prelude=1,
                       n_recurrent=1, n_coda=1, recurrent_steps=4)
    tr = RegMachineTrainer(cfg, ArithmeticTokenizer(), mcfg,
                           program_coef=pc, answer_coef=1.0)
    t0 = time.time()
    for s in range(cfg.steps):
        tr.train_step(s)
        if (s+1) % 1000 == 0:
            r = tr.evaluate(512)
            print(f"[{arm}] {s+1:5d} ({time.time()-t0:4.0f}s) answer {r['answer_acc']:.3f} "
                  f"canonical {r['canonical_acc']:.3f} instr {r['instr_acc']:.3f}", flush=True)

### 5c. The studies that were never run

`d2g2` (5.2e8 expressions) and `d3g1` (1.3e10) are large enough that memorisation is
unavailable. Both arms include `coconut-long`, the wall-clock-matched control that
retracted the structure claim on `d2g1` — so these are the honest versions.

In [ ]:
%env LAMB_DEVICE=cuda
!python -m lamb.study --task d2g2 --seeds 5 --workers 1 --out /content/d2g2.json
!python -m lamb.study --task d3g1 --seeds 5 --workers 1 --out /content/d3g1.json

### 5d. What does the encoder's prior knowledge actually buy?

The experiment the frozen-encoder design makes cheap, and which nobody runs: the same
latent core against a pretrained encoder vs one that has only seen generated text.
The difference prices the encoder's benchmark exposure directly, instead of arguing
about whether "frozen" means "uncontaminated". Needs the GSM8K/GSM1K caching step
first (see `docs/ROADMAP.md` on the bridge).

In [ ]:
from lamb.bridge import extract_quantities, Resampler
# the peripheral is exact and needs no training: quantities come out by rule
p = "Weng earns $12.50 per hour. Yesterday she did 50 minutes of babysitting."
print([(q.value, q.scale, q.approx) for q in extract_quantities(p)])
print("resampler is length-invariant:", Resampler(768, 512, n_latents=32)(
    __import__("torch").randn(2, 400, 768)).shape)